In [1]:
import requests
import pandas as pd
import io

In [ ]:
url = "https://esploradati.istat.it/SDMXWS/rest/data/41_983"
headers_csv = {'Accept': 'application/vnd.sdmx.data+csv;version=1.0.0'}

r = requests.get(url, headers=headers_csv, timeout=120)
print(r.status_code)

if r.status_code == 200:
    with open('istat_raw.csv', 'w', encoding='utf-8') as f:
        f.write(r.text)
    print("Salvato su disco, non serve richiamare l'API di nuovo")

# l'API SDMX di ISTAT ha un limite di 5 chiamate al minuto: se lo superi,
# l'IP viene bloccato per 24-48 ore (un meccanismo un po' becero, a dirla
# come l'ha detto il prof)

# dato che i dati storici sugli incidenti non cambiano in tempo reale,
# non ha senso interrogare l'API ogni volta che eseguo il notebook,
# quindi faccio la chiamata UNA sola volta e appena arriva la risposta
# la salvo subito su disco (istat_raw.csv). Da qui in avanti lavoro
# sempre su quella copia locale, senza più toccare l'endpoint perché
# meno richieste = meno rischio di blocco, e anche un po' di pietà
# per l'infrastruttura non esattamente performante di ISTAT

In [2]:
import pandas as pd

df = pd.read_csv('istat_raw.csv')
df.info()

# visualizzo le informazioni del dataframe, ci sono 573.552 righe e 16 colonne ma 9 di queste
# sono completamente vuote (0 non-null):
# OBS_STATUS, NOTE_DS, NOTE_REF_AREA, NOTE_DATA_TYPE, NOTE_RESULT,
# NOTE_TIME_PERIOD, BASE_PER, UNIT_MEAS, UNIT_MULT
# quindi le rimuoverò in fase di pulizia

<class 'pandas.DataFrame'>
RangeIndex: 573552 entries, 0 to 573551
Data columns (total 16 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   DATAFLOW          573552 non-null  str    
 1   FREQ              573552 non-null  str    
 2   REF_AREA          573552 non-null  int64  
 3   DATA_TYPE         573552 non-null  str    
 4   RESULT            573552 non-null  str    
 5   TIME_PERIOD       573552 non-null  int64  
 6   OBS_VALUE         573552 non-null  int64  
 7   OBS_STATUS        0 non-null       float64
 8   NOTE_DS           0 non-null       float64
 9   NOTE_REF_AREA     0 non-null       float64
 10  NOTE_DATA_TYPE    0 non-null       float64
 11  NOTE_RESULT       0 non-null       float64
 12  NOTE_TIME_PERIOD  0 non-null       float64
 13  BASE_PER          0 non-null       float64
 14  UNIT_MEAS         0 non-null       float64
 15  UNIT_MULT         0 non-null       float64
dtypes: float64(9), int64(3), str(4)

In [3]:
print(df['REF_AREA'].astype(str).str.len().unique())

print(df['DATA_TYPE'].unique())
print(df['RESULT'].unique())

# ho controllato quante cifre hanno i codici REF_AREA convertendoli in stringa
# e misurandone la lunghezza: sono uscite lunghezze diverse (4, 5, 6) invece
# di un valore uniforme. Questo succede perché la colonna è stata letta da
# pandas come numero (int64), e i numeri non mantengono gli zeri davanti,
# quindi un codice come 001001 viene letto e salvato come 1001 (4 cifre).
# i codici comune ISTAT dovrebbero invece essere sempre a 6 cifre, quindi
# qui c'è un problema da correggere prima di usare questa colonna per
# fare il join con i dati SITUAS più avanti

# ho controllato anche i valori unici (.unique()) di DATA_TYPE e RESULT,
# per capire cosa contengono queste due colonne prima di
# usarle nell'analisi:
# DATA_TYPE ha 2 valori: KILLINJ, ROADACC
# RESULT ha 3 valori: F, M, 9
# da questi nomi non è ancora chiaro il significato esatto (es. cosa
# rappresenta il valore 9), serve un controllo più approfondito più avanti

[4 5 6]
<StringArray>
['KILLINJ', 'ROADACC']
Length: 2, dtype: str
<StringArray>
['F', 'M', '9']
Length: 3, dtype: str


In [4]:
df['REF_AREA'] = df['REF_AREA'].astype(str).str.zfill(6)

print(df['REF_AREA'].head(10))
print(df['REF_AREA'].astype(str).str.len().unique())

# prima avevo trovato che REF_AREA aveva lunghezze diverse (4, 5, 6 cifre)
# perché pandas l'aveva letta come numero, perdendo gli zeri davanti ai
# codici comune. qui la converto in stringa con zfill(6), che aggiunge
# zeri a sinistra finché la stringa non arriva a 6 caratteri, così i
# codici tornano tutti uniformi e nel formato giusto per essere
# confrontati/uniti più avanti con i dati SITUAS

# controllo che il fix abbia funzionato: stampo le prime righe e
# ricontrollo le lunghezze. il risultato [6] conferma che ora tutti
# i codici hanno 6 cifre

0    001001
1    001001
2    001001
3    001001
4    001001
5    001001
6    001001
7    001001
8    001001
9    001001
Name: REF_AREA, dtype: str
[6]


In [5]:
df[df['DATA_TYPE'] == 'ROADACC']['OBS_VALUE'].describe()

# guardando i numeri di .describe() si vede che qualcosa non è regolare:
# la mediana (50%) è 3, ma la media è quasi 25 (molto più alta della
# mediana). anche la deviazione standard (257) è enorme, circa 10 volte
# la media stessa. quando media e mediana sono così distanti, e la
# deviazione standard è più grande della media, di solito vuol dire che
# c'è almeno un valore molto più alto degli altri che tira su la media
# rispetto al resto dei dati, cioè un possibile outlier

count    191184.000000
mean         24.997599
std         257.519973
min           0.000000
25%           1.000000
50%           3.000000
75%          12.000000
max       23135.000000
Name: OBS_VALUE, dtype: float64

In [ ]:
df[(df['DATA_TYPE'] == 'ROADACC') & (df['OBS_VALUE'] == 23135)]

# vado a vedere a quale riga appartiene il valore più alto (23135, il max
# uscito da .describe()), per capire se è un errore nei dati o un valore
# legittimo
# il codice REF_AREA che esce è 058091, che cercando su internet, corrisponde a Roma Capitale:
# essendo il comune più popoloso d'Italia, ha senso che abbia molti più
# incidenti rispetto agli altri comuni solo per via del traffico e delle
# dimensioni

,DATAFLOW,FREQ,REF_AREA,DATA_TYPE,RESULT,TIME_PERIOD,OBS_VALUE,OBS_STATUS,NOTE_DS,NOTE_REF_AREA,NOTE_DATA_TYPE,NOTE_RESULT,NOTE_TIME_PERIOD,BASE_PER,UNIT_MEAS,UNIT_MULT
348636,IT1:41_983(1.0),A,058091,ROADACC,9,2004,23135,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
print(df['FREQ'].unique())
print(df['DATAFLOW'].unique())
df = df.drop(columns=['OBS_STATUS', 'NOTE_DS', 'NOTE_REF_AREA', 'NOTE_DATA_TYPE',
                       'NOTE_RESULT', 'NOTE_TIME_PERIOD', 'BASE_PER', 'UNIT_MEAS', 'UNIT_MULT', 'FREQ', 'DATAFLOW'])

# controllo i valori unici di FREQ e DATAFLOW: entrambe hanno un solo
# valore (FREQ='A', cioè frequenza annuale; DATAFLOW='IT1:41_983(1.0)',
# l'identificativo del dataset). non sono colonne vuote come le 9 di
# prima, ma essendo sempre lo stesso valore su tutte le righe non
# distinguono niente, quindi non aggiungono informazione utile.
# le tolgo, assieme alle colonne coi valori null, sarebbe inutile tenerle visto che non hanno dati

<StringArray>
['A']
Length: 1, dtype: str
<StringArray>
['IT1:41_983(1.0)']
Length: 1, dtype: str


In [11]:
df.info()

# controllo con .info() che il drop abbia funzionato: ora il dataframe
# ha 5 colonne invece di 16, tutte piene al 100% (573.552 valori
# non-null su ognuna), e la memoria occupata è scesa da 70 MB a 22 MB

<class 'pandas.DataFrame'>
RangeIndex: 573552 entries, 0 to 573551
Data columns (total 5 columns):
 #   Column       Non-Null Count   Dtype
---  ------       --------------   -----
 0   REF_AREA     573552 non-null  str  
 1   DATA_TYPE    573552 non-null  str  
 2   RESULT       573552 non-null  str  
 3   TIME_PERIOD  573552 non-null  int64
 4   OBS_VALUE    573552 non-null  int64
dtypes: int64(2), str(3)
memory usage: 21.9 MB


In [ ]:
print(df[(df['DATA_TYPE'] == 'KILLINJ') & (df['RESULT'] == 'F')]['OBS_VALUE'].describe())
print(df[(df['DATA_TYPE'] == 'KILLINJ') & (df['RESULT'] == 'M')]['OBS_VALUE'].describe())
print(df[(df['DATA_TYPE'] == 'KILLINJ') & (df['RESULT'] == '9')]['OBS_VALUE'].describe())

# faccio lo stesso controllo .describe() già fatto per ROADACC, ma
# questa volta sui feriti (RESULT=F) e sui morti (RESULT=M):
# - feriti: media 35, mediana 5, max 30254 -> stessa forma fortemente
#   asimmetrica vista per ROADACC, pochi casi enormi tirano su la media
# - morti: media 0.53, mediana 0 -> più della metà dei comuni/anni ha
#   zero morti, ha senso perché morire in un incidente è un evento raro.
#   il massimo però è 363, un numero alto, da verificare prima di darlo per buono.
# cercando online ho trovato che ISTAT usa convenzionalmente M per morti ed F per feriti (confermato anche
# da un report ufficiale ISTAT sugli incidenti stradali):
# https://www.istat.it/wp-content/uploads/2025/12/focus-piemonte-valle-d-aosta-2024.pdf
# sempre cercando online, il codice 9 per indicare "non classificato"
# in varie classificazioni, per esempio quella Ateco delle attività economiche:
# https://www.istat.it/it/files/2022/03/volume_integrale_ATECO2007.pdf

count    191184.000000
mean         35.091985
std         338.109881
min           0.000000
25%           1.000000
50%           5.000000
75%          18.000000
max       30254.000000
Name: OBS_VALUE, dtype: float64
count    191184.000000
mean          0.533225
std           2.783505
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max         363.000000
Name: OBS_VALUE, dtype: float64
count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: OBS_VALUE, dtype: float64


In [13]:
print(df[(df['DATA_TYPE'] == 'KILLINJ') & (df['RESULT'] == 'F') & (df['OBS_VALUE'] == 30254)])
print(df[(df['DATA_TYPE'] == 'KILLINJ') & (df['RESULT'] == 'M') & (df['OBS_VALUE'] == 363)])

# controllo a chi appartengono i due valori massimi trovati prima
# (30254 feriti, 363 morti): in entrambi i casi esce lo stesso comune,
# REF_AREA=058091 quindi Roma Capitale, lo steso identico comune che era
# già uscito come massimo per ROADACC. quindi è lo stesso fenomeno di
# prima: essendo il comune più popoloso e con più traffico d'Italia,
# ha senso che abbia i numeri più alti in assoluto su tutte le metriche

       REF_AREA DATA_TYPE RESULT  TIME_PERIOD  OBS_VALUE
348588   058091   KILLINJ      F         2004      30254
       REF_AREA DATA_TYPE RESULT  TIME_PERIOD  OBS_VALUE
348610   058091   KILLINJ      M         2002        363


In [ ]:
df.groupby('TIME_PERIOD').size()

# conto quante righe ci sono per ogni anno, per vedere se la copertura
# è uniforme nel tempo o se ci sono buchi. dal 2001 al 2023 il calo è
# lento e graduale (da 24303 a 23703 righe, circa -2.5% in 22 anni) -
# plausibile, dato che negli ultimi decenni l'Italia ha accorpato diversi
# piccoli comuni. ma dal 2023 al 2024 il calo è molto più brusco: da
# 23703 a 19017 righe, circa -20% in un solo anno, troppo per essere
# spiegato da normali fusioni di comuni, serve un controllo più approfondito

TIME_PERIOD
2001    24303
2002    24306
2003    24306
2004    24300
2005    24303
2006    24303
2007    24303
2008    24303
2009    24300
2010    24282
2011    24276
2012    24276
2013    24276
2014    24177
2015    24141
2016    24000
2017    23934
2018    23862
2019    23745
2020    23712
2021    23712
2022    23712
2023    23703
2024    19017
dtype: int64

In [ ]:
print(df[df['TIME_PERIOD'] == 2023]['REF_AREA'].nunique())
print(df[df['TIME_PERIOD'] == 2024]['REF_AREA'].nunique())

# verifico se il calo visto sopra nel conteggio delle righe per anno
# riguarda intere righe di comuni mancanti, o solo singole metriche: conto
# i comuni distinti (REF_AREA) per il 2023 e il 2024. sono 7901
# comuni nel 2023, e solo 6339 nel 2024. una perdita di 1562 comuni interi,
# circa il 20%, come già notato nel conteggio totale delle righe.
# questo conferma che il 2024 ha dati probabilmente incompleti
# (mancano intere rilevazioni comunali), non solo "meno incidenti
# registrati" e per questo più avanti escluderò questo anno dal dataset princiale

7901
6339


In [ ]:
df_completo = df.copy()

# tengo una copia con tutti gli anni (df_completo), incluso il 2024,
# nel caso mi serva in futuro per un grafico isolato di trend con
# l'avviso "dato parziale"

df = df[df['TIME_PERIOD'] != 2024]

# escludo il 2024 dal dataset principale (df).
# cercando online ho trovato che la regione Friuli-Venezia Giulia non ha
# mandato a ISTAT i dati dettagliati delle Polizie Locali per il 2024,
# quindi quei dati sono stati ricostruiti solo a livello aggregato e non
# comune per comune. in più, il dettaglio a livello di singolo comune è
# anche l'ultimo a essere pubblicato rispetto ai dati regionali, quindi
# ha senso che manchino un bel po' di comuni proprio nel 2024. se lo
# tenessi nelle analisi aggregate rischierei di far sembrare più sicuri
# dei comuni che in realtà non hanno semplicemente il dato, non che non
# hanno avuto incidenti.
# fonte: https://www.istat.it/wp-content/uploads/2025/12/focus-lazio-2024.pdf